# 08 — Heatmap Visualization

Visualizes ML predictions as a heatmap overlaid on the Manhattan grid.

**Input:** `csv/07_predictions.csv`

**Output:**
- `outputs/latest/08_heatmap_predictions.png` — static matplotlib heatmap
- `outputs/latest/08_heatmap_interactive.html` — interactive folium Leaflet.js map

In [ ]:
# ── Papermill parameters ──────────────────────────────
PLOTS_DIR = "outputs/latest"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import json
import os

os.makedirs(PLOTS_DIR, exist_ok=True)

with open("grid.json", encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]

# Load predictions
df = pd.read_csv("csv/07_predictions.csv", dtype={"cell_id": str})
print(f"Loaded {len(df)} cell predictions")
print(f"Commercial: {(df['predicted_zone'] == 'Commercial').sum()}")
print(f"Residential: {(df['predicted_zone'] == 'Residential').sum()}")

In [ ]:
# ── Compute cell rectangle dimensions in degrees ─────
import math

REF_LAT = df["cell_lat"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
HALF_LAT = LAT_STEP / 2
HALF_LON = LON_STEP / 2

print(f"Cell size: {LAT_STEP:.6f} lat x {LON_STEP:.6f} lon")
print(f"Lat range: {df['cell_lat'].min():.4f} - {df['cell_lat'].max():.4f}")
print(f"Lon range: {df['cell_lon'].min():.4f} - {df['cell_lon'].max():.4f}")

In [ ]:
# ── Static matplotlib heatmap ─────────────────────────

# Custom diverging colormap: blue (Residential) ↔ red (Commercial)
cmap = LinearSegmentedColormap.from_list(
    "res_com", ["#2166AC", "#D1E5F0", "#FDDBC7", "#B2182B"], N=256)

fig, ax = plt.subplots(figsize=(8, 18))

for _, row in df.iterrows():
    rect = mpatches.Rectangle(
        (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
        LON_STEP, LAT_STEP,
        linewidth=0.1, edgecolor="gray",
        facecolor=cmap(row["prob_commercial"]),
        alpha=0.85,
    )
    ax.add_patch(rect)

ax.set_xlim(df["cell_lon"].min() - 0.005, df["cell_lon"].max() + 0.005)
ax.set_ylim(df["cell_lat"].min() - 0.005, df["cell_lat"].max() + 0.005)
ax.set_aspect("equal")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Manhattan Grid — Commercial Probability\n({len(df)} cells, {CELL_SIZE_M}m grid)", fontsize=14)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label("P(Commercial)", fontsize=11)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(["Residential", "0.25", "0.50", "0.75", "Commercial"])

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_heatmap_predictions.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/08_heatmap_predictions.png")

In [ ]:
# ── Interactive folium map ─────────────────────────────
try:
    import folium
    from folium import Rectangle
    import branca.colormap as bcm

    m = folium.Map(
        location=[REF_LAT, df["cell_lon"].mean()],
        zoom_start=13,
        tiles="CartoDB positron",
    )

    # Color scale
    colormap = bcm.LinearColormap(
        colors=["#2166AC", "#D1E5F0", "#FDDBC7", "#B2182B"],
        vmin=0, vmax=1,
        caption="P(Commercial)"
    )

    for _, row in df.iterrows():
        bounds = [
            [row["cell_lat"] - HALF_LAT, row["cell_lon"] - HALF_LON],
            [row["cell_lat"] + HALF_LAT, row["cell_lon"] + HALF_LON],
        ]
        color = colormap(row["prob_commercial"])
        popup_text = (f"<b>{row['cell_id']}</b><br>"
                      f"Actual: {row['zone_type']}<br>"
                      f"Predicted: {row['predicted_zone']}<br>"
                      f"P(Commercial): {row['prob_commercial']:.2f}")
        Rectangle(
            bounds=bounds,
            color="gray", weight=0.3,
            fill=True, fill_color=color, fill_opacity=0.75,
            popup=folium.Popup(popup_text, max_width=200),
        ).add_to(m)

    colormap.add_to(m)

    html_path = f"{PLOTS_DIR}/08_heatmap_interactive.html"
    m.save(html_path)
    print(f"Saved: {html_path}")
    print("Open in browser for interactive exploration.")

except ImportError:
    print("folium not installed — skipping interactive map.")
    print("Install with: pip install folium")

In [ ]:
# ── Summary statistics ────────────────────────────────
print("Prediction summary:")
print(f"  Total cells: {len(df)}")
print(f"  Predicted Commercial: {(df['predicted_zone'] == 'Commercial').sum()} "
      f"({100 * (df['predicted_zone'] == 'Commercial').mean():.1f}%)")
print(f"  Predicted Residential: {(df['predicted_zone'] == 'Residential').sum()} "
      f"({100 * (df['predicted_zone'] == 'Residential').mean():.1f}%)")
print(f"  Mean P(Commercial): {df['prob_commercial'].mean():.3f}")
print(f"  Median P(Commercial): {df['prob_commercial'].median():.3f}")

# Accuracy vs actual labels
correct = (df["predicted_zone"] == df["zone_type"]).sum()
# Note: zone_type still has original labels; need to map for comparison
_map = {"Commercial": "Commercial", "Mixed-Use": "Residential", "Residential": "Residential"}
df["actual_binary"] = df["zone_type"].map(_map)
accuracy = (df["predicted_zone"] == df["actual_binary"]).mean()
print(f"\n  Overall accuracy (on all cells): {accuracy:.3f}")